# 20_GameSkeleton — Khung xương game oẳn tù tì (chỉ test luật, chưa làm giao diện đẹp)

Model: `step2 s1` (`Data_321/checkpoints_task9/step2/seed1/TCN.pt`). Chỉ chạy model, **không train**.

**Cách chạy:** kích hoạt `C:\envs\train-gpu\Scripts\activate.bat`, mở notebook bằng kernel `train-gpu`, chạy lần lượt cell 1 → 7. Cell 5 tự kiểm tra luật game (không cần webcam). Cell 7 mở webcam.

**Phím:** `SPACE` chơi | `r` reset điểm | `q` thoát.

**Luật:** SPACE → đếm 3, 2, 1 → "ra". Mất tay quá `lost_max` giây khi đang đếm thì dừng, nhắc, có tay lại thì đếm lại từ 3. Tại "ra" AI chốt đáp án ngay (lớp có xác suất cao nhất, trung bình `K` khung gần nhất) rồi ra quân khắc chế. Sau "ra" có `judge_sec` giây: khi một lớp đạt ngưỡng `thr` liên tục `stable_n` khung thì chốt cử chỉ người chơi và phân thắng/thua; hết giờ mà chưa chốt được thì "không xác định" (không tính điểm). Mọi tham số nằm ở `Game(...)` trong cell 4.

In [8]:
#cell 1: import, đường dẫn, kiểm tra file cần có, bật CUDA
import time, math, collections
import numpy as np, torch, cv2
import torch.nn as nn, torch.nn.functional as F
import mediapipe as mp
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook này không chạy trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
TASK = BASE / "hand_landmarker.task"
CKPT = BASE / "checkpoints_task9" / "step2" / "seed1" / "TCN.pt"      # step2 s1
for p in (TASK, CKPT):
    assert p.exists(), f"Không thấy: {p}"
CLASSES = ["paper", "rock", "scissors"]                              # nhãn 0, 1, 2
print("OK | GPU:", torch.cuda.get_device_name(0))

OK | GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [9]:
#cell 2: TCN + 546 đặc trưng (giống 13/19), nạp checkpoint step2 s1, làm nóng GPU
class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__(); self.pad = 2 * d
        self.conv = nn.Conv1d(c, c, 3, dilation=d); self.drop = nn.Dropout(0.1)
    def forward(self, x):
        return x + self.drop(F.relu(self.conv(F.pad(x, (self.pad, 0)))))

class TCN(nn.Module):
    def __init__(self, in_features=126, c=64):
        super().__init__(); self.inp = nn.Linear(in_features, c)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        self.out = nn.Linear(c, 3)
    def forward(self, x):
        return self.out(self.blocks(self.inp(x).transpose(1, 2)).transpose(1, 2))

PAIR_I, PAIR_J = torch.triu_indices(21, 21, offset=1, device=dev)    # 210 cặp khớp

def features_from_pos(pos, hand_mask):
    """pos (b,T,21,3) -> (b,T,546): 126 gốc + 210 khoảng cách + 210 vận tốc khoảng cách (giống 13)."""
    pos = torch.where(hand_mask[:, :, None, None], pos, torch.zeros_like(pos))
    b, T = pos.shape[:2]
    flat = pos.reshape(b, T, 63)
    pv = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    vel = torch.zeros_like(flat); vel[:, 1:] = (flat[:, 1:] - flat[:, :-1]) * pv
    dist = torch.linalg.vector_norm(pos[:, :, PAIR_I] - pos[:, :, PAIR_J], dim=-1) * hand_mask[:, :, None]
    dvel = torch.zeros_like(dist); dvel[:, 1:] = (dist[:, 1:] - dist[:, :-1]) * pv
    return torch.cat([flat, vel, dist, dvel], dim=-1)

def geo_from_base(X, hand_mask):
    return features_from_pos(X[..., :63].reshape(X.shape[0], X.shape[1], 21, 3), hand_mask)

net = TCN(in_features=546).to(dev)
ck = torch.load(CKPT, map_location=dev, weights_only=False)
assert ck.get("cfg", {}).get("n_in") == 546, f"Checkpoint không phải họ step2: {ck.get('cfg')}"
net.load_state_dict(ck["w"]); net.eval()
with torch.no_grad(): net(torch.zeros(1, 128, 546, device=dev))
print(f"Đã nạp step2 s1 | epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f}")

Đã nạp step2 s1 | epoch 27 | val_loss 0.3224


In [10]:
#cell 3: MediaPipe (CPU), nội suy webcam -> 120 fps, hàm trả xác suất 3 lớp của khung cuối (giống 13/19)
landmarker = mp.tasks.vision.HandLandmarker.create_from_options(mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(TASK)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO, num_hands=1))
T0 = time.perf_counter()          # timestamp MediaPipe phải luôn tăng, kể cả khi chạy lại cell 7

def detect(frame_bgr, ts_ms):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    res = landmarker.detect_for_video(mp.Image(mp.ImageFormat.SRGB, rgb), ts_ms)
    if not res.hand_landmarks:
        return None
    return np.array([[p.x, p.y, p.z] for p in res.hand_landmarks[0]], np.float32)

FPS_MODEL, WIN = 120, 128
buf = collections.deque(maxlen=WIN)
state = {"prev_lm": None, "prev_t": None}

def push(lm, t):
    prev, pt = state["prev_lm"], state["prev_t"]
    n = 1 if pt is None else max(1, round((t - pt) * FPS_MODEL))
    if lm is None:
        buf.extend([np.full((21, 3), np.nan, np.float32)] * n)
    elif prev is None:
        buf.append(lm)
    else:
        buf.extend(prev + (lm - prev) * (k / n) for k in range(1, n + 1))
    state["prev_lm"], state["prev_t"] = lm, t

def base_features(lm):
    has = ~np.isnan(lm[:, 0, 0])
    q = lm - lm[:, :1]
    q = np.nan_to_num(q / np.maximum(np.linalg.norm(q[:, 9], axis=1)[:, None, None], 1e-8)).astype(np.float32)
    q[~has] = 0
    pos = q.reshape(len(q), 63)
    vel = np.zeros_like(pos)
    if len(pos) > 1:
        vel[1:] = (pos[1:] - pos[:-1]) * (has[1:] & has[:-1])[:, None]
    return np.hstack([pos, vel]).astype(np.float32), has

def probs_now():
    """Xác suất (paper, rock, scissors) của khung cuối; None nếu khung cuối mất tay."""
    lm = np.stack(buf)
    if np.isnan(lm[-1, 0, 0]):
        return None
    base, has = base_features(lm)
    xb, mb = torch.from_numpy(base)[None].to(dev), torch.from_numpy(has)[None].to(dev)
    with torch.no_grad():
        return net(geo_from_base(xb, mb)).softmax(-1)[0, -1].cpu().numpy()
print("Sẵn sàng: MediaPipe + nội suy 120 fps + hàm probs_now()")

Sẵn sàng: MediaPipe + nội suy 120 fps + hàm probs_now()


In [11]:
#cell 4: luật game (máy trạng thái, không dính tới webcam nên test được riêng ở cell 5)
COUNTER = {0: 2, 1: 0, 2: 1}      # người ra paper -> AI ra scissors; rock -> paper; scissors -> rock

class Game:
    def __init__(self, K=1, count_sec=1.0, lost_max=0.3, judge_sec=1.5, stable_n=5, thr=0.6, result_sec=2.0):
        self.K, self.count_sec, self.lost_max = K, count_sec, lost_max
        self.judge_sec, self.stable_n, self.thr, self.result_sec = judge_sec, stable_n, thr, result_sec
        self.score = {"ai": 0, "player": 0, "draw": 0}
        self.state, self.msg = "WAIT", "Nhan SPACE de choi"
        self.number = 3
        self.guess = self.ai = self.player = self.outcome = None

    def start(self, t, probs):
        """Bấm SPACE (chỉ có tác dụng khi đang chờ). Chưa có tay thì chờ tay rồi mới đếm."""
        if self.state != "WAIT":
            return
        self.guess = self.ai = self.player = self.outcome = None
        if probs is None:
            self.state, self.msg = "NEED_HAND", "Dua tay vao khung hinh"
        else:
            self._begin_count(t, probs)

    def _begin_count(self, t, probs):
        self.state, self.msg, self.number = "COUNT", "", 3
        self.t0, self.lost_since = t, None
        self.hist = collections.deque([probs], maxlen=self.K)     # K khung có tay gần nhất

    def step(self, t, probs):
        """Gọi mỗi khung webcam. probs: mảng 3 xác suất, hoặc None nếu mất tay."""
        if self.state == "NEED_HAND":
            if probs is not None:
                self._begin_count(t, probs)                       # có tay lại -> đếm lại từ 3
        elif self.state == "COUNT":
            self._count(t, probs)
        elif self.state == "JUDGE":
            self._judge(t, probs)
        elif self.state == "RESULT" and t - self.t_res >= self.result_sec:
            self.state, self.msg = "WAIT", "Nhan SPACE de choi"

    def _count(self, t, probs):
        if probs is None:
            self.lost_since = self.lost_since if self.lost_since is not None else t
            if t - self.lost_since > self.lost_max:               # mất tay quá lâu -> dừng đếm
                self.state, self.msg = "NEED_HAND", "Mat tay! Dua tay vao khung hinh"
                return
        else:
            self.lost_since = None
            self.hist.append(probs)
        e = t - self.t0
        if e < 3 * self.count_sec:
            self.number = 3 - int(e / self.count_sec)
            return
        p = np.mean(self.hist, axis=0)                            # "ra": chốt đáp án ngay, không random
        self.guess, self.ai = int(p.argmax()), COUNTER[int(p.argmax())]
        self.state, self.msg, self.t_ra = "JUDGE", "RA!", t
        self.run_cls, self.run_n = None, 0

    def _judge(self, t, probs):
        if probs is not None and probs.max() >= self.thr:
            c = int(probs.argmax())
            self.run_n = self.run_n + 1 if c == self.run_cls else 1
            self.run_cls = c
        else:
            self.run_cls, self.run_n = None, 0
        if self.run_n >= self.stable_n:
            self._finish(t, self.run_cls)
        elif t - self.t_ra >= self.judge_sec:
            self._finish(t, None)

    def _finish(self, t, player):
        self.player, self.t_res, self.state = player, t, "RESULT"
        d = None if player is None else (self.ai - player) % 3
        self.outcome = "unknown" if d is None else ("draw", "player", "ai")[d]
        if d is not None:
            self.score[self.outcome] += 1
print("Đã định nghĩa Game.")

Đã định nghĩa Game.


In [12]:
#cell 5: tự kiểm tra luật game bằng dữ liệu giả (không cần webcam)
rock, low, dt = np.array([.05, .90, .05]), np.array([.40, .35, .25]), 1 / 30

def run(g, probs, n):
    for _ in range(n):
        run.t += dt; g.step(run.t, probs)

def until(g, probs, state, n=200):
    for _ in range(n):
        run.t += dt; g.step(run.t, probs)
        if g.state == state: return
    raise AssertionError(f"Không tới được {state}, đang ở {g.state}")

# 1) đếm đủ 3 giây -> AI đoán rock, ra paper; người giữ rock -> AI thắng
run.t = 0.0; g = Game(); g.start(run.t, rock)
until(g, rock, "JUDGE"); assert run.t > 2.9 and g.guess == 1 and g.ai == 0, (run.t, g.guess, g.ai)
run(g, rock, 10); assert g.state == "RESULT" and g.outcome == "ai" and g.score["ai"] == 1

# 2) mất tay ngắn (0.2 s) vẫn đếm tiếp; mất lâu thì dừng, có tay lại thì đếm lại từ 3
run.t = 0.0; g = Game(); g.start(run.t, rock)
run(g, None, 6); assert g.state == "COUNT"
run(g, None, 10); assert g.state == "NEED_HAND"
run(g, rock, 1); assert g.state == "COUNT" and g.number == 3

# 3) sau "ra" 1.5 s mà không đạt ngưỡng -> không xác định, không tính điểm
run.t = 0.0; g = Game(); g.start(run.t, rock)
until(g, rock, "JUDGE"); run(g, low, 60)
assert g.state == "RESULT" and g.outcome == "unknown" and sum(g.score.values()) == 0

# 4) bấm SPACE khi chưa có tay -> chờ tay
run.t = 0.0; g = Game(); g.start(run.t, None); assert g.state == "NEED_HAND"
print("Cả 4 phép thử luật game đều đạt.")

Cả 4 phép thử luật game đều đạt.


In [13]:
#cell 6: giao diện tối thiểu (chữ đen trên nền màu, chữ không dấu vì OpenCV không vẽ được dấu)
COLORS = {0: (60, 60, 255), 1: (255, 150, 40), 2: (60, 210, 60)}      # BGR: paper đỏ, rock xanh dương, scissors xanh lá
GRAY, YELLOW, BLACK = (190, 190, 190), (150, 255, 255), (0, 0, 0)
BONES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
         (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]
RESULT_TXT = {"ai": "AI THANG", "player": "BAN THANG", "draw": "HOA", "unknown": "Khong nhan duoc cu chi - choi lai"}

def put_label(img, text, x, y, bg, scale=0.8):
    (w, h), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    cv2.rectangle(img, (x, y - h - base - 6), (x + w + 10, y + 2), bg, -1)
    cv2.putText(img, text, (x + 5, y - base), cv2.FONT_HERSHEY_SIMPLEX, scale, BLACK, 2, cv2.LINE_AA)

def draw_hand(img, lm, color, mirror=True):
    h, w = img.shape[:2]
    px, py = (lm[:, 0] * w).astype(int), (lm[:, 1] * h).astype(int)
    if mirror: px = w - 1 - px
    for a, b in BONES:
        cv2.line(img, (px[a], py[a]), (px[b], py[b]), color, 2, cv2.LINE_AA)
    cv2.rectangle(img, (px.min() - 12, py.min() - 12), (px.max() + 12, py.max() + 12), color, 2)

def draw_game(img, g):
    h, w = img.shape[:2]
    if g.state == "COUNT":
        put_label(img, str(g.number), w // 2 - 30, h // 2 + 40, YELLOW, 4)
    if g.state in ("JUDGE", "RESULT"):
        put_label(img, f"RA! AI doan ban ra {CLASSES[g.guess]} -> AI ra {CLASSES[g.ai]}", 10, h - 90, COLORS[g.ai], 0.7)
    put_label(img, RESULT_TXT[g.outcome] if g.state == "RESULT" else g.msg, 10, h - 50, YELLOW, 0.9)
    s = g.score
    put_label(img, f"AI {s['ai']} - {s['player']} Nguoi | hoa {s['draw']}", 10, h - 10, YELLOW, 0.6)
print("Đã định nghĩa giao diện tối thiểu.")

Đã định nghĩa giao diện tối thiểu.


In [14]:
#cell 7: tạo file âm thanh (tick cho 3-2-1, ra khác hẳn) và hàm phát không chặn
import wave, winsound
SND = BASE / "sounds"; SND.mkdir(exist_ok=True)

def make_wav(path, freq, ms, vol=0.6):
    sr = 44100; t = np.arange(int(sr * ms / 1000)) / sr
    x = np.sin(2 * np.pi * freq * t) * np.minimum(1, np.minimum(t, t[-1] - t) * 200)   # mờ đầu/cuối, tránh tiếng "tạch"
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes((x * vol * 32767).astype(np.int16).tobytes())

SOUNDS = {"tick": (SND / "tick.wav", 800, 120), "ra": (SND / "ra.wav", 1400, 350)}   # tick: thấp, ngắn | ra: cao, dài
for path, f, ms in SOUNDS.values():
    if not path.exists(): make_wav(path, f, ms)                                       # có rồi thì không tạo lại

def play(name):
    winsound.PlaySound(str(SOUNDS[name][0]), winsound.SND_FILENAME | winsound.SND_ASYNC)
print("Âm thanh ở:", SND)

Âm thanh ở: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\sounds


In [18]:
#cell 8: chạy webcam + game — SPACE: chơi | r: reset điểm | q: thoát
CAM_ID, MIRROR = 0, True
cap = cv2.VideoCapture(CAM_ID, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480); cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
assert cap.isOpened(), "Không mở được webcam (thử đổi CAM_ID sang 1)."

buf.clear(); state.update(prev_lm=None, prev_t=None)
g = Game()                                    # đổi tham số luật ở đây, ví dụ Game(K=2, lost_max=0.5)
fps, last = 0.0, time.perf_counter()
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        now = time.perf_counter()
        lm = detect(frame, int((now - T0) * 1000))
        push(lm, now)
        pr = probs_now() if lm is not None else None
        before = (g.state, g.number)
        g.step(now, pr)
        if g.state == "COUNT" and (g.state, g.number) != before: play("tick")   # nhịp 3, 2, 1 (kể cả khi đếm lại)
        elif before[0] == "COUNT" and g.state == "JUDGE": play("ra")           # nhịp "ra"                       # luật chạy trước khi vẽ nên đáp án AI hiện ngay ở khung "ra"

        disp = cv2.flip(frame, 1) if MIRROR else frame
        fps, last = 0.9 * fps + 0.1 / max(now - last, 1e-6), now
        if pr is not None:
            c, conf = int(pr.argmax()), float(pr.max())
            col = COLORS[c] if conf >= g.thr else GRAY
            draw_hand(disp, lm, col, MIRROR)
            put_label(disp, f"{CLASSES[c]}{'' if conf >= g.thr else '?'} {conf:.0%}", 10, 70, col, 0.6)
        put_label(disp, f"FPS {fps:.0f}", 10, 30, YELLOW, 0.6)
        draw_game(disp, g)
        cv2.imshow("RPS game skeleton (SPACE: choi | r: reset | q: thoat)", disp)

        k = cv2.waitKey(1) & 0xFF
        if k == ord("q"): break
        if k == 32:
            g.start(now, pr)
            if g.state == "COUNT": play("tick")      # nhịp "3" ngay lúc bấm
        if k == ord("r"): g.score = {"ai": 0, "player": 0, "draw": 0}
finally:
    cap.release(); cv2.destroyAllWindows()
print("Đã đóng webcam.")

Đã đóng webcam.
